In [7]:
import requests

WIKI_API_URL = "https://en.wikipedia.org/w/api.php" #JA:https://ja.wikipedia.org/w/api.php

headers = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36"}

# 検索キーワードに該当するWikipediaページを取得する関数
def fetch_search_results(keyword, limit=5):
    params = {
        "action": "query",
        "list": "search",
        "srsearch": keyword,
        "format": "json",
        "srlimit": limit
    }

    try:
        response = requests.get(WIKI_API_URL, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        return data.get("query", {}).get("search", [])
    except requests.RequestException as e:
        print("検索リクエストでエラーが発生しました:", e)
        return []

# 指定されたタイトルのページ内容を取得する関数
def fetch_page_content(title):
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True
    }

    try:
        response = requests.get(WIKI_API_URL, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        page = next(iter(data.get("query", {}).get("pages", {}).values()), {})
        return page.get("extract", "ページ内容が見つかりません。")
    except requests.RequestException as e:
        print("ページ内容取得でエラーが発生しました:", e)
        return "ページ内容が取得できませんでした。"

In [8]:
# テストコードで確認
keyword = "Python"

# キーワードに該当するページ一覧を取得
search_results = fetch_search_results(keyword)
if search_results:
    print("【キーワードに該当するページ一覧】")
    for result in search_results[:3]:
        print(f"- {result['title']}")

    # 最初のページタイトルを取得
    first_title = search_results[0]["title"]
    print("\n【最初のページタイトル】")
    print(first_title)

    # 最初のページの内容を取得
    page_content = fetch_page_content(first_title)
    print("\n【ページ内容】")
    print(page_content[:500])
else:
    print("検索結果が見つかりません。")

【キーワードに該当するページ一覧】
- Python (programming language)
- Python
- Monty Python

【最初のページタイトル】
Python (programming language)

【ページ内容】
Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, "plain English" naming, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.
Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Pyth


In [9]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"


In [10]:
# 要約を行うプロンプトを作成
prompt = f"""
Please summarize.

# Condition：
- Even if elementary school students read it, they can understand it.
- 300 words or less.

# Source text：
{page_content[:1000]}
"""

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "user", "content": prompt},
    ],
    max_tokens=500,
    temperature=0.3
)

In [11]:
print(response.choices[0].message.content.strip())


Python is a programming language that is easy to read and write. It was created by Guido van Rossum in the late 1980s as a new version of an older language called ABC. Python is known for its simple rules, which make it friendly for beginners and fun to use.

One of the cool things about Python is that it uses indentation, which means you need to line up your code neatly. This helps people understand the code better. Python also has a lot of built-in tools and libraries, which are like helpful boxes of tools that make programming easier.

Python can be used in different ways, but it mainly focuses on something called object-oriented programming. This means it helps you organize your code in a way that makes sense, like putting similar things together.

In 2008, a big update called Python 3.0 was released. This update changed some things, so not all old Python code would work with it. Since then, new versions have come out regularly. The latest versions that are supported include Python